# Meta-Learning with MAML on a Custom Pendulum Task

This notebook demonstrates the implementation of Model-Agnostic Meta-Learning (MAML) on a custom meta-learning benchmark created from the classic 'Pendulum-v1' environment. 

The goal of meta-learning is to train a model that can quickly adapt to new tasks. Here, we create different 'tasks' by varying the physical parameters (gravity) of the Pendulum environment. The MAML algorithm will learn a set of initial parameters that can be effectively fine-tuned for any new pendulum task with only a few gradient steps.

## 1. Setup and Imports

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from functools import partial

# Add project root to path to import our modules
import sys
sys.path.append('..')

from meta_rl.algorithms.maml import MAML, ActorCritic
from meta_rl.utils.data_utils import collect_trajectories, process_trajectories

# Configure JAX to use CPU for reproducibility if no GPU is present
try:
    jax.devices('gpu')
except RuntimeError:
    jax.config.update('jax_platform_name', 'cpu')

## 2. Creating a Meta-Learning Environment

In [ ]:
class MetaPendulumEnv:
    """
    A meta-environment that creates Pendulum tasks with varying gravity.
    """
    def __init__(self, num_tasks):
        self.tasks = np.random.uniform(5.0, 15.0, size=num_tasks)
        self.num_tasks = num_tasks

    def sample_task(self):
        """Sample a random task (gravity value)."""
        return np.random.choice(self.tasks)

    def get_env(self, task):
        """Get a gymnasium environment for a given task."""
        env = gym.make('Pendulum-v1', g=task)
        return env

# Create a meta-environment with 20 different gravity settings
meta_env = MetaPendulumEnv(num_tasks=20)

## 3. MAML Training Loop

In [ ]:
# Hyperparameters
META_ITERATIONS = 100
INNER_UPDATES = 1
META_BATCH_SIZE = 10 # Number of tasks per meta-update
TRAJECTORIES_PER_TASK = 20
MAX_STEPS_PER_TRAJECTORY = 200
INNER_LR = 0.01
META_LR = 0.001

# Setup environment and MAML agent
dummy_env = meta_env.get_env(meta_env.sample_task())
obs_dim = dummy_env.observation_space.shape[0]
action_dim = dummy_env.action_space.shape[0]

maml = MAML(
    action_dim=action_dim,
    obs_dim=obs_dim,
    inner_lr=INNER_LR,
    meta_lr=META_LR
)

# Initialize parameters
rng = jax.random.PRNGKey(0)
rng, key = jax.random.split(rng)
meta_params, opt_state = maml.init_params(key)

policy_fn = maml.network.apply
losses = []

print("Starting MAML training...")
for meta_iter in range(META_ITERATIONS):
    # Prepare a batch of tasks
    support_batches = []
    query_batches = []
    for _ in range(META_BATCH_SIZE):
        task = meta_env.sample_task()
        env = meta_env.get_env(task)
        
        # Collect data for inner update (support set)
        rng, key = jax.random.split(rng)
        support_trajectories = collect_trajectories(env, meta_params, policy_fn, TRAJECTORIES_PER_TASK, MAX_STEPS_PER_TRAJECTORY, key, action_scale=2.0)
        support_batch = process_trajectories(support_trajectories, policy_fn, meta_params)
        support_batches.append(support_batch)
        
        # Collect data for meta-update (query set)
        rng, key = jax.random.split(rng)
        query_trajectories = collect_trajectories(env, meta_params, policy_fn, TRAJECTORIES_PER_TASK, MAX_STEPS_PER_TRAJECTORY, key, action_scale=2.0)
        query_batch = process_trajectories(query_trajectories, policy_fn, meta_params)
        query_batches.append(query_batch)
    
    # Stack the batches for vmap compatibility
    stacked_support_batch = jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *support_batches)
    stacked_query_batch = jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *query_batches)
    stacked_meta_batch = (stacked_support_batch, stacked_query_batch)
    
    # Meta-update
    meta_params, opt_state, loss = maml.outer_update(meta_params, opt_state, stacked_meta_batch)
    losses.append(loss)
    
    if (meta_iter + 1) % 10 == 0:
        print(f"Meta-iteration {meta_iter + 1}/{META_ITERATIONS}, Loss: {loss:.4f}")

print("Training finished.")

## 4. Visualizing Results and Meta-Testing

In [ ]:
# Plot the training loss
plt.figure(figsize=(10, 5))
plt.plot(losses)
plt.title('MAML Meta-Loss over Training')
plt.xlabel('Meta-Iteration')
plt.ylabel('Loss')
plt.grid(True)
plt.show()

### Meta-Test: Adapting to a New Task

Now we'll test our meta-trained agent on a completely new task it has never seen before.

In [ ]:
def evaluate_policy(env, params, policy_fn, num_episodes=10, action_scale=1.0):
    total_rewards = []
    for _ in range(num_episodes):
        obs, _ = env.reset()
        episode_reward = 0
        for _ in range(MAX_STEPS_PER_TRAJECTORY):
            mu, _, _ = policy_fn({'params': params}, obs[None, ...])
            action = np.array([mu.squeeze()]) * action_scale # Use deterministic action for eval
            obs, reward, terminated, truncated, _ = env.step(action)
            episode_reward += reward
            if terminated or truncated:
                break
        total_rewards.append(episode_reward)
    return np.mean(total_rewards)

# Create a new, unseen task
new_task_gravity = 20.0 # A much higher gravity
test_env = gym.make('Pendulum-v1', g=new_task_gravity)

# 1. Evaluate the meta-policy directly (before adaptation)
reward_before_adaptation = evaluate_policy(test_env, meta_params, policy_fn, action_scale=2.0)
print(f"Average reward on new task (before adaptation): {reward_before_adaptation:.2f}")

# 2. Adapt the policy with a few gradient steps
rng, key = jax.random.split(rng)
test_trajectories = collect_trajectories(test_env, meta_params, policy_fn, 10, MAX_STEPS_PER_TRAJECTORY, key, action_scale=2.0)
test_batch = process_trajectories(test_trajectories, policy_fn, meta_params)
adapted_params = maml.inner_update(meta_params, test_batch)

# 3. Evaluate the adapted policy
reward_after_adaptation = evaluate_policy(test_env, adapted_params, policy_fn, action_scale=2.0)
print(f"Average reward on new task (after adaptation): {reward_after_adaptation:.2f}")